# CUDA Demo Validation

Validasi startup corpus penuh dan flow demo Gradio pada GPU. Pilih **Runtime > Change runtime type > T4 GPU**, lalu jalankan **Run all**. Notebook akan mengunduh artefak `cuda_demo_validation.json` jika seluruh pemeriksaan berhasil.

In [ ]:
import torch

assert torch.cuda.is_available(), "Pilih Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
BRANCH = "dev/apiip"

%cd /content
!test -d indonesian-legal-compliance-rag || git clone --depth 1 --branch {BRANCH} https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git
!git -C indonesian-legal-compliance-rag pull --ff-only origin {BRANCH}
%cd /content/indonesian-legal-compliance-rag

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y -q torchvision torchcodec
!python scripts/check_environment.py
!python -m unittest discover -s tests -v

In [ ]:
!python -m gdown --folder https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql -O data/raw

from pathlib import Path
from src.rag import REGULATION_BY_FILE

corpus_dir = Path("data/raw")
missing = [name for name in REGULATION_BY_FILE if not (corpus_dir / name).is_file()]
assert not missing, f"PDF corpus tidak lengkap: {missing}"
print(f"Corpus OK: {len(REGULATION_BY_FILE)} PDF")

In [ ]:
import json
import platform
import subprocess
from datetime import datetime, timezone

import torch

from app import NO_SOURCES, answer_question, build_app, build_search

search = build_search(corpus_dir, "cuda")
demo = build_app(search)
assert demo.config["components"] and demo.config["dependencies"]

cases = [
    {
        "name": "answerable",
        "question": "Apa bentuk perizinan usaha berisiko menengah rendah?",
        "expected_status": "answer",
    },
    {
        "name": "unanswerable",
        "question": "Apa izin menurut PP Nomor 28 Tahun 2025?",
        "expected_status": "insufficient_context",
    },
]
results = []
for case in cases:
    messages, status, elapsed, sources, debug = answer_question(
        case["question"], [], search
    )
    assert debug.get("final_status") == case["expected_status"], debug
    if case["expected_status"] == "answer":
        assert debug["retrieved"] and sources != NO_SOURCES
    else:
        assert not debug["retrieved"]
    results.append(
        {
            **case,
            "status_markdown": status,
            "elapsed_markdown": elapsed,
            "answer_markdown": messages[-1]["content"],
            "source_cards_markdown": sources,
            "debug": debug,
        }
    )
    print(f"{case['name']}: {debug['final_status']} ({elapsed})")

artifact = {
    "passed": True,
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
    "corpus_files": sorted(REGULATION_BY_FILE),
    "ui": {
        "components": len(demo.config["components"]),
        "dependencies": len(demo.config["dependencies"]),
    },
    "results": results,
}
output = Path("eval/results/demo-validation/cuda_demo_validation.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps({"passed": True, "artifact": str(output)}, indent=2))

In [ ]:
from google.colab import files

files.download(str(output))